In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/kumarajarshi/life-expectancy-who/Life Expectancy Data.csv


### Read in Data

In [2]:
data = pd.read_csv('/kaggle/input/datasets/kumarajarshi/life-expectancy-who/Life Expectancy Data.csv')
df = pd.DataFrame(data)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2938 entries, 0 to 2937
Data columns (total 22 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Country                          2938 non-null   object 
 1   Year                             2938 non-null   int64  
 2   Status                           2938 non-null   object 
 3   Life expectancy                  2928 non-null   float64
 4   Adult Mortality                  2928 non-null   float64
 5   infant deaths                    2938 non-null   int64  
 6   Alcohol                          2744 non-null   float64
 7   percentage expenditure           2938 non-null   float64
 8   Hepatitis B                      2385 non-null   float64
 9   Measles                          2938 non-null   int64  
 10   BMI                             2904 non-null   float64
 11  under-five deaths                2938 non-null   int64  
 12  Polio               

### Standardize Country and Status

In [3]:
df.columns = (df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace('-', '_', regex=False)
    .str.replace(r'[^\w\s]', '', regex=True)
)

df['country'] = df['country'].astype(str)
df['status'] = df['status'].astype(str)

df['country'] = df['country'].str.strip().str.capitalize()
df['status'] = df['status'].str.strip().str.capitalize()

print(df.head())


       country  year      status  life_expectancy  adult_mortality  \
0  Afghanistan  2015  Developing             65.0            263.0   
1  Afghanistan  2014  Developing             59.9            271.0   
2  Afghanistan  2013  Developing             59.9            268.0   
3  Afghanistan  2012  Developing             59.5            272.0   
4  Afghanistan  2011  Developing             59.2            275.0   

   infant_deaths  alcohol  percentage_expenditure  hepatitis_b  measles  ...  \
0             62     0.01               71.279624         65.0     1154  ...   
1             64     0.01               73.523582         62.0      492  ...   
2             66     0.01               73.219243         64.0      430  ...   
3             69     0.01               78.184215         67.0     2787  ...   
4             71     0.01                7.097109         68.0     3013  ...   

   polio  total_expenditure  diphtheria  hivaids         gdp  population  \
0    6.0              

### Find Duplicates

In [4]:
duplicated = df[df.duplicated()]
print(duplicated)

Empty DataFrame
Columns: [country, year, status, life_expectancy, adult_mortality, infant_deaths, alcohol, percentage_expenditure, hepatitis_b, measles, bmi, under_five_deaths, polio, total_expenditure, diphtheria, hivaids, gdp, population, thinness__1_19_years, thinness_5_9_years, income_composition_of_resources, schooling]
Index: []

[0 rows x 22 columns]


### Find/Fill Null/NaN

In [5]:
to_zero_cols = ['alcohol',
                'hepatitis_b',
                 'polio',
                 'diphtheria',
                 'schooling',
                 'total_expenditure'
                ]
df[to_zero_cols] = df[to_zero_cols].fillna(0)
df = df.fillna({'life_expectancy': df['life_expectancy'].mean(),
                'adult_mortality': df['adult_mortality'].mean(),
                'bmi': df['bmi'].mean(),
                'gdp' : df['gdp'].mean(),
                'population' : df['population'].mean(),
                'income_composition_of_resources' : df['income_composition_of_resources'].mean()})
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2938 entries, 0 to 2937
Data columns (total 22 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   country                          2938 non-null   object 
 1   year                             2938 non-null   int64  
 2   status                           2938 non-null   object 
 3   life_expectancy                  2938 non-null   float64
 4   adult_mortality                  2938 non-null   float64
 5   infant_deaths                    2938 non-null   int64  
 6   alcohol                          2938 non-null   float64
 7   percentage_expenditure           2938 non-null   float64
 8   hepatitis_b                      2938 non-null   float64
 9   measles                          2938 non-null   int64  
 10  bmi                              2938 non-null   float64
 11  under_five_deaths                2938 non-null   int64  
 12  polio               

### Make New Columns

In [6]:
df['deaths_from_disease'] = df['alcohol'] + df['hepatitis_b'] + df['measles'] + df['polio'] + df['diphtheria'] + df['hivaids']
df['thinness_in_children'] = df['thinness_5_9_years'] + df['thinness__1_19_years']

del df['thinness_5_9_years']
del df['thinness__1_19_years']

print(df.head())



       country  year      status  life_expectancy  adult_mortality  \
0  Afghanistan  2015  Developing             65.0            263.0   
1  Afghanistan  2014  Developing             59.9            271.0   
2  Afghanistan  2013  Developing             59.9            268.0   
3  Afghanistan  2012  Developing             59.5            272.0   
4  Afghanistan  2011  Developing             59.2            275.0   

   infant_deaths  alcohol  percentage_expenditure  hepatitis_b  measles  ...  \
0             62     0.01               71.279624         65.0     1154  ...   
1             64     0.01               73.523582         62.0      492  ...   
2             66     0.01               73.219243         64.0      430  ...   
3             69     0.01               78.184215         67.0     2787  ...   
4             71     0.01                7.097109         68.0     3013  ...   

   polio  total_expenditure  diphtheria  hivaids         gdp  population  \
0    6.0              

### Data Wrangling

In [7]:
year2015 = df.loc[df['year'] == 2015]
col_2015 = year2015['life_expectancy'].reset_index()
del col_2015['index']
year2000 = df.loc[df['year'] == 2000]
col_2000 = year2000['life_expectancy'].reset_index()
del col_2000['index']

df['life_expectancy_change'] = ((col_2015 - col_2000)/col_2000) *100

def has_preventable_disease(row):
    if row['measles'] > 1:
        return 'Yes'
    if row['polio'] > 1:
        return 'Yes'
    else:
        return 'No'

df['has_preventable_diseases'] = df.apply(has_preventable_disease, axis = 1)

results = df.groupby('year').agg({'life_expectancy' : ['mean', 'min', 'max']})